## KDIGO AKI Labeling and Outcome Prevalence

### 1. Data Loading and Cohort Definition

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import datetime, json, hashlib
import gc
import sys
sys.path.append(str(Path.cwd()))
from kdigo_pipeline import (
    compute_measurement_gaps, kdigo_scr_15x, kdigo_scr_03, kdigo_scr_03_early,
    combine_scr_labels, get_time_window, kdigo_urine_elapsed_time,
    generate_urine_labels_elapsed_time
)

In [ ]:
PROJECT_DIR = Path.cwd()

DATA = PROJECT_DIR / "Mimic"
HOSP = DATA / "hosp"
ICU = DATA / "icu"

OUTPUT = PROJECT_DIR / "processed_outputs"
DERIVED_DIR = DATA / "derived"


Sensitivity_OUTPUT = PROJECT_DIR/"processed_outputs/Sensitivity_Analysis"
OUTPUT.mkdir(exist_ok=True)
Sensitivity_OUTPUT.mkdir(exist_ok=True)


In [ ]:
cohort = pd.read_parquet(OUTPUT / "cohort.parquet")

print(cohort.shape)
cohort.head()


In [ ]:
print("cohort rows:", len(cohort))
print("unique stays:", cohort["stay_id"].nunique())
print(cohort["stay_id"].duplicated().sum())

In [ ]:
assert cohort["stay_id"].is_unique
cohort_stay_ids = set(cohort["stay_id"])

In [ ]:
labevents_raw = pd.read_csv(DATA / "hosp/labevents.csv", usecols=["subject_id", "hadm_id", "itemid","charttime","valuenum"], parse_dates=["charttime"])

# Step 1: standard hadm_id match
labevents_with_hadm = labevents_raw.merge(
    cohort[["subject_id", "hadm_id", "stay_id", "intime"]],
    on=["subject_id", "hadm_id"], how="inner"
)

# Step 2: null-hadm_id rows, matched by subject_id + pre-admission time window
labevents_no_hadm = labevents_raw[labevents_raw["hadm_id"].isna()].merge(
    cohort[["subject_id", "hadm_id", "stay_id", "intime"]].rename(columns={"hadm_id": "cohort_hadm_id"}),
    on="subject_id", how="inner"
)
labevents_no_hadm["hours_before_intime"] = (
    labevents_no_hadm["intime"] - labevents_no_hadm["charttime"]
).dt.total_seconds() / 3600
labevents_no_hadm = labevents_no_hadm[
    (labevents_no_hadm["hours_before_intime"] >= 0) &
    (labevents_no_hadm["hours_before_intime"] <= 168)
].drop(columns=["cohort_hadm_id", "hours_before_intime"])

# Step 3: de-duplicate multi-stay matches (keep closest stay in time, if subject has >1 stay)
dup_matches = labevents_no_hadm.groupby(["subject_id","charttime","itemid"]).size()
print("Null-hadm_id lab rows matching multiple stays:", (dup_matches > 1).sum())
labevents_no_hadm["time_to_intime"] = (labevents_no_hadm["intime"] - labevents_no_hadm["charttime"])
labevents_no_hadm = labevents_no_hadm.sort_values("time_to_intime").drop_duplicates(
    subset=["subject_id","charttime","itemid"], keep="first"
).drop(columns=["time_to_intime"])

labevents = pd.concat([labevents_with_hadm, labevents_no_hadm], ignore_index=True)
print("Labevents with hadm_id match:", len(labevents_with_hadm))
print("Labevents with null hadm_id, recovered:", len(labevents_no_hadm))
print("Cohort-restricted labevents (combined):", len(labevents))

In [ ]:
# Confirm no remaining duplicates after the drop_duplicates step
remaining_dups = labevents_no_hadm.groupby(["subject_id","charttime","itemid"]).size()
print("Remaining multi-stay matches after dedup:", (remaining_dups > 1).sum())
print("Total null-hadm_id rows after dedup:", len(labevents_no_hadm))

In [ ]:
ambiguous_subjects = labevents_no_hadm[
    labevents_no_hadm.duplicated(subset=["subject_id","charttime","itemid"], keep=False)
]["subject_id"].nunique()
print("Distinct subjects involved in ambiguous null-hadm_id matches:", ambiguous_subjects)

In [ ]:
print(labevents_no_hadm["intime"].sub(labevents_no_hadm["charttime"]).dt.total_seconds().div(3600).describe())

In [ ]:
URINE_OUTPUT_ITEMIDS = [ 226557,
    226558,
    226559,
    226560,
    226561,
    226563,
    226564,
    226565,
    226567,
    226584,
    227488,
    227489
]
output_chunks = []
for chunk in pd.read_csv( DATA / "icu/outputevents.csv",usecols=[ "subject_id","hadm_id", "stay_id", "charttime","itemid","value"],chunksize=500_000):

    chunk = chunk[chunk["itemid"].isin(URINE_OUTPUT_ITEMIDS)]

    if chunk.empty:
        continue
    chunk = chunk[chunk["stay_id"].isin(cohort_stay_ids)]
    if not chunk.empty:
        output_chunks.append(chunk)
outputevents = pd.concat(output_chunks,ignore_index=True)

del output_chunks
gc.collect()

outputevents["charttime"] = pd.to_datetime( outputevents["charttime"],errors="coerce")
outputevents = outputevents.merge(cohort[["stay_id", "intime"]],on="stay_id",how="inner")
outputevents = compute_measurement_gaps(outputevents)
print("Cohort-restricted outputevents:", len(outputevents))

In [ ]:
WEIGHT_ITEMIDS = [ 226512,  
    224639,
    226531  ] 
    

weight_chunks = []

for chunk in pd.read_csv(DATA / "icu/chartevents.csv",usecols=["stay_id","charttime", "itemid","valuenum"],chunksize=500_000):

    chunk = chunk[chunk["itemid"].isin(WEIGHT_ITEMIDS)]
    if chunk.empty:
        continue
    chunk = chunk[chunk["stay_id"].isin(cohort_stay_ids)]
    if not chunk.empty:
        weight_chunks.append(chunk)

chartevents = pd.concat(weight_chunks,ignore_index=True)

del weight_chunks
gc.collect()

chartevents["charttime"] = pd.to_datetime(chartevents["charttime"],errors="coerce")
chartevents = chartevents.merge( cohort[["stay_id", "intime"]],on="stay_id",how="inner")

print("Cohort-restricted weight chartevents:", len(chartevents))
print("Weight-event stays:", chartevents["stay_id"].nunique())

In [ ]:
lb_check = chartevents[chartevents["itemid"] == 226531]
print("Rows with lb-only admission weight itemid:", len(lb_check))
print("Distinct stays:", lb_check["stay_id"].nunique())

In [ ]:
kg_stays = set(chartevents.loc[chartevents["itemid"] == 226512, "stay_id"])
lb_stays = set(chartevents.loc[chartevents["itemid"] == 226531, "stay_id"])

print("Stays with kg record:", len(kg_stays))
print("Stays with lb record:", len(lb_stays))
print("Stays with BOTH kg and lb:", len(kg_stays & lb_stays))
print("Stays with lb ONLY (no kg at all):", len(lb_stays - kg_stays))
print("Stays with kg ONLY (no lb):", len(kg_stays - lb_stays))

In [ ]:
d_items = pd.read_csv(ICU/"d_items.csv")
d_icd = pd.read_csv(HOSP / "d_icd_diagnoses.csv", dtype={"icd_code": str})

In [ ]:
urine_item_summary = (outputevents.groupby("itemid").agg( n=("value", "size"),min_value=("value", "min"),max_value=("value", "max")).reset_index().merge(d_items[["itemid", "label", "abbreviation"]],on="itemid", how="left").sort_values("itemid"))

urine_item_summary

In [ ]:
outputevents["urine_ml"] = pd.to_numeric(outputevents["value"],errors="coerce")

In [ ]:

# Require same-timestamp pairing of 227488/227489 before applying sign adjustment,
# to avoid large negative net values from unpaired irrigant entries.
irrigant_pairs = outputevents[outputevents["itemid"].isin([227488, 227489])]
paired_ts = irrigant_pairs.groupby(["stay_id","charttime"])["itemid"].apply(
    lambda x: set(x) == {227488, 227489}
)
paired_keys = set(paired_ts[paired_ts].index)

def is_paired_irrigant(row):
    return (row["stay_id"], row["charttime"]) in paired_keys

outputevents["irrigant_paired"] = outputevents.apply(
    lambda r: is_paired_irrigant(r) if r["itemid"] in [227488, 227489] else True, axis=1
)

# Only apply the sign adjustment where properly paired; otherwise treat as missing (drop the row)
outputevents.loc[(outputevents["itemid"]==227488) & (~outputevents["irrigant_paired"]), "urine_ml"] = np.nan
outputevents = outputevents.dropna(subset=["urine_ml"])

outputevents["urine_ml"] = np.where(
    (outputevents["itemid"] == 227488) & (outputevents["urine_ml"] > 0),
    -1 * outputevents["urine_ml"],
    outputevents["urine_ml"]
)

In [ ]:
NON_IRRIGANT_ITEMIDS = [226557, 226558, 226559, 226560, 226561, 226563, 226564, 226565, 226567, 226584]

# Drop negative values for standard urine-collection items (never physiologically valid)
invalid_negative_mask = (outputevents["itemid"].isin(NON_IRRIGANT_ITEMIDS)) & (outputevents["urine_ml"] < 0)
print(f"Dropping {invalid_negative_mask.sum()} invalid negative non-irrigant rows")
outputevents = outputevents[~invalid_negative_mask].copy()

In [ ]:
print("Negative urine_ml rows(irrigation adjustments):", (outputevents["urine_ml"] < 0).sum())
print("Duplicate rows:", outputevents.duplicated(subset=["stay_id","charttime","itemid"]).sum())

In [ ]:

# Cohort restricted weightsource availability
kg_stays_cohort=set(chartevents.loc[chartevents["itemid"]==226512,"stay_id"])
lb_stays_cohort=set(chartevents.loc[chartevents["itemid"]==226531,"stay_id"])
daily_weight_cohort=set(chartevents.loc[chartevents["itemid"]==224639,"stay_id"])

print("Cohort size:", cohort["stay_id"].nunique())
print("Cohort stays with kg record:", len(kg_stays_cohort))
print("Cohort stays with lb record:", len(lb_stays_cohort))
print("Cohort stays with daily weight record:", len(daily_weight_cohort))
print("Cohort stays with BOTH kg and lb:", len(kg_stays_cohort & lb_stays_cohort))
print("Cohort stays with lb ONLY (no kg at all):", len(lb_stays_cohort - kg_stays_cohort))
print("Cohort stays with kg ONLY (no lb):", len(kg_stays_cohort - lb_stays_cohort))

In [ ]:

print("Cohort LOS stats (hours):")
print((cohort["outtime"] - cohort["intime"]).dt.total_seconds().div(3600).describe())

print("\nStays with LOS < 24h:", ((cohort["outtime"] - cohort["intime"]).dt.total_seconds()/3600 < 24).sum())
print("Stays with LOS < 48h:", ((cohort["outtime"] - cohort["intime"]).dt.total_seconds()/3600 < 48).sum())
print("Stays with missing outtime:", cohort["outtime"].isna().sum())

cohort["los_hours"] = (cohort["outtime"] - cohort["intime"]).dt.total_seconds() / 3600
short_stay_ids = set(cohort.loc[cohort["los_hours"] < 48, "stay_id"])
print("\nShort-stay (LOS<48h) stay_ids captured:", len(short_stay_ids))

In [ ]:
diagnoses = pd.read_csv(HOSP / "diagnoses_icd.csv",
                        usecols=["hadm_id", "icd_code", "icd_version"],
                        dtype={"icd_code": str})
dx = diagnoses.copy()
dx["icd_code"] = dx["icd_code"].str.strip().str.upper()

# ESRD: ICD-9 585.6 / ICD-10 N18.6
esrd_mask = (((dx["icd_version"] == 9)  & (dx["icd_code"] == "5856")) |
             ((dx["icd_version"] == 10) & (dx["icd_code"].str.startswith("N186", na=False))))
# Chronic dialysis dependence: ICD-9 V45.1/V45.11, ICD-10 Z99.2
dial_mask = (((dx["icd_version"] == 9)  & dx["icd_code"].isin(["V451", "V4511"])) |
             ((dx["icd_version"] == 10) & (dx["icd_code"] == "Z992")))

esrd_hadm_ids     = set(dx.loc[esrd_mask, "hadm_id"])
dialysis_hadm_ids = set(dx.loc[dial_mask, "hadm_id"])

esrd_stay_ids          = set(cohort.loc[cohort["hadm_id"].isin(esrd_hadm_ids), "stay_id"])
dialysis_stay_ids      = set(cohort.loc[cohort["hadm_id"].isin(dialysis_hadm_ids), "stay_id"])
esrd_dialysis_stay_ids = esrd_stay_ids | dialysis_stay_ids      # used everywhere downstream

print("Cohort stays with ESRD code:", len(esrd_stay_ids))
print("Cohort stays with chronic-dialysis code:", len(dialysis_stay_ids))
print("Overlap:", len(esrd_stay_ids & dialysis_stay_ids))
print("Union (esrd_dialysis_stay_ids):", len(esrd_dialysis_stay_ids))

### 2.Baseline Creatinine Values Calculation

In [ ]:
CREATININE=50912
creatinine=labevents[labevents["itemid"] == CREATININE].copy()

In [ ]:
print(creatinine.shape)
creatinine.head()

In [ ]:
creatinine=creatinine.sort_values(["stay_id", "charttime"])

In [ ]:
# calculates the time difference between creatinine measurement and the ICU intime(in hours)
creatinine["hours_from_icu"]=(creatinine["charttime"]-creatinine["intime"]).dt.total_seconds() / 3600

In [ ]:
#Baseline serum creatinine is defined as the lowest value measured within the 7 days preceding ICU admission 
# 1.creatinine values of upto 7 days(168 hours) before icu admission.
baseline_window = creatinine[(creatinine["hours_from_icu"] >= -168) &(creatinine["hours_from_icu"] < 0)]

In [ ]:
# 2.least value of the last 7 days creatinine values for each stay
baseline_scr = (baseline_window.groupby("stay_id")["valuenum"].min().reset_index().rename(columns={"valuenum":"baseline_scr"}))

In [ ]:
print(baseline_window)

In [ ]:
# baseline creatinie is calculated for each ICU stay
print(baseline_scr)

In [ ]:
# if no pre-ICU measurement is available, the first serum creatinine recorded after ICU admission will be considered as the baseline serum creatinine value
missing_baseline = (cohort[["stay_id"]].merge( baseline_scr,on="stay_id", how="left"))

In [ ]:
print(missing_baseline)

In [ ]:
# gets the first creatinine measuerd after icu admission
first_scr=(creatinine[ creatinine["hours_from_icu"] >= 0].sort_values("charttime").groupby("stay_id")
.first()[["valuenum"]].rename(columns={ "valuenum":"first_scr"}).reset_index())

In [ ]:
baseline_scr = (missing_baseline.merge( first_scr,on="stay_id",how="left"))

In [ ]:
print(baseline_scr)

In [ ]:

baseline_scr["pre_icu_baseline"] = baseline_scr["baseline_scr"]

In [ ]:
#if baseline_scr is Nan, then fill it with the first_scr value
baseline_scr["baseline_scr"] = (baseline_scr["baseline_scr"].fillna(baseline_scr["first_scr"]))

In [ ]:
print(baseline_scr)

### 3.KDIGO Serum Criteria Label Generation

#### Early(0-24 hrs) Assessment

In [ ]:

scr_0_24 = get_time_window(creatinine, 0, 24)

In [ ]:
early_scr_15x = kdigo_scr_15x(scr_0_24,baseline_scr)
early_scr_03 = kdigo_scr_03_early(scr_0_24)
early_scr_labels = combine_scr_labels(cohort,early_scr_15x,early_scr_03)

In [ ]:
print(early_scr_labels["aki_scr"].value_counts())
print("Early AKI:",early_scr_labels["aki_scr"].sum())

In [ ]:
print("tm_since_last" in outputevents.columns)
print(outputevents["tm_since_last"].isna().sum())

### 4. KDIGO Urine Output Criteria

#### Urine Output Processing Functions

In [ ]:
def aggregate_hourly_output(df):
    """
    Aggregates urine output to ICU-hour level.
    All urine-output measurements within the same ICU hour are summed before plausibility filtering.
    """
    temp = df.copy()
    temp["urine_ml"] = pd.to_numeric(temp["urine_ml"],errors="coerce")
    temp = temp.dropna(subset=["urine_ml"])
    temp["hour"] = np.floor(temp["hours_from_icu"]).astype(int)

    hourly = (temp.groupby(["stay_id", "hour"], as_index=False).agg(urine_ml=("urine_ml", "sum")))
    # Bound the summed hourly total
    hourly = hourly[hourly["urine_ml"].between(0, 2000)].copy()

    return hourly

#### ICU Weight Determination

In [ ]:
d_items[d_items["label"].str.contains( "weight",case=False,na=False)][["itemid","label"]]

In [ ]:
WEIGHT_ITEMID_KG = 226512# Admission Weight (Kg)
WEIGHT_ITEMID_KG_DAILY = 224639# Daily Weight (Kg)
WEIGHT_ITEMID_LB = 226531# Admission Weight (Lbs.)

In [ ]:
w_raw = chartevents[chartevents["itemid"].isin([WEIGHT_ITEMID_KG, WEIGHT_ITEMID_KG_DAILY, WEIGHT_ITEMID_LB])].copy()
w_raw["hours_from_icu"] = (w_raw["charttime"] - w_raw["intime"]).dt.total_seconds() / 3600
w_raw = w_raw[(w_raw["hours_from_icu"] >= 0) & (w_raw["hours_from_icu"] < 24)].copy()


In [ ]:
#convert lb to kg
w_raw["weight_kg"] = np.where(w_raw["itemid"] == WEIGHT_ITEMID_LB,pd.to_numeric(w_raw["valuenum"], errors="coerce") * 0.45359237,pd.to_numeric(w_raw["valuenum"], errors="coerce"))
w_raw["source_itemid"] = w_raw["itemid"]


In [ ]:
# Keep only valid numeric weights
w_raw = w_raw.dropna(subset=["weight_kg"])

# Plausibility filtering
w_raw = w_raw[w_raw["weight_kg"].between(20, 300)].copy()

# kg-native is always preferred over lb-converted regardless of charttime order
w_raw["priority"] = np.where(w_raw["itemid"].isin([WEIGHT_ITEMID_KG, WEIGHT_ITEMID_KG_DAILY]), 0, 1)
w_raw = w_raw.sort_values(["stay_id", "priority", "charttime"])
weight = (w_raw.groupby("stay_id", as_index=False).first()[["stay_id", "weight_kg", "source_itemid", "charttime"]])

print("Usable first-24h weights:", weight["stay_id"].nunique())
print(weight["source_itemid"].map({
    WEIGHT_ITEMID_KG: "kg-native (admission)",
    WEIGHT_ITEMID_KG_DAILY: "kg-native (daily)",
    WEIGHT_ITEMID_LB: "lb-converted"
}).value_counts())

In [ ]:

#eeconciliation for stays with BOTH kg-native and lb-native within 24h, compare converted values to verify conversion correctness
both_units_raw = chartevents[chartevents["itemid"].isin([WEIGHT_ITEMID_KG, WEIGHT_ITEMID_KG_DAILY, WEIGHT_ITEMID_LB])].copy()
both_units_raw["hours_from_icu"] = (both_units_raw["charttime"] - both_units_raw["intime"]).dt.total_seconds() / 3600
both_units_raw = both_units_raw[(both_units_raw["hours_from_icu"] >= 0) & (both_units_raw["hours_from_icu"] < 24)]

# plausibility filtering
both_units_raw_filtered = both_units_raw[
    (both_units_raw["itemid"].isin([WEIGHT_ITEMID_KG, WEIGHT_ITEMID_KG_DAILY]) & both_units_raw["valuenum"].between(20, 300)) |
    ((both_units_raw["itemid"] == WEIGHT_ITEMID_LB) &both_units_raw["valuenum"].between(44.09, 661.39) ) # 20-300 kg expressed in lbs
].copy()

print("Rows before plausibility filter:", len(both_units_raw))
print("Rows after plausibility filter:", len(both_units_raw_filtered))

kg_stay_ids = set(both_units_raw_filtered.loc[both_units_raw_filtered["itemid"].isin([WEIGHT_ITEMID_KG, WEIGHT_ITEMID_KG_DAILY]), "stay_id"])
lb_stay_ids = set(both_units_raw_filtered.loc[both_units_raw_filtered["itemid"] == WEIGHT_ITEMID_LB, "stay_id"])
overlap_ids = kg_stay_ids & lb_stay_ids

kg_first = (both_units_raw_filtered[both_units_raw_filtered["itemid"].isin([WEIGHT_ITEMID_KG, WEIGHT_ITEMID_KG_DAILY]) & both_units_raw_filtered["stay_id"].isin(overlap_ids)]
            .sort_values("charttime").groupby("stay_id")["valuenum"].first())
lb_first_converted = (both_units_raw_filtered[(both_units_raw_filtered["itemid"] == WEIGHT_ITEMID_LB) & both_units_raw_filtered["stay_id"].isin(overlap_ids)]
                       .sort_values("charttime").groupby("stay_id")["valuenum"].first() * 0.45359237)

conversion_check = (kg_first - lb_first_converted).dropna()
print(f"Stays with both kg and lb in first 24h: {len(overlap_ids)}")
print("kg vs lb-converted difference:")
print(conversion_check.describe())

In [ ]:

weight_durations_check = pd.read_csv(DATA/ "derived" / "weight_durations.csv", parse_dates=["starttime","endtime"])
ref_weight_stays = set(weight_durations_check["stay_id"].unique()) & set(cohort["stay_id"])
own_weight_stays = set(weight["stay_id"])

print(f"Cohort size: {cohort['stay_id'].nunique()}")
print(f"Stays covered by mimiciv_derived.weight_durations: {len(ref_weight_stays)}")
print(f"Stays covered by first-24h weight table: {len(own_weight_stays)}")
print(f"Overlap: {len(ref_weight_stays & own_weight_stays)}")

In [ ]:
weight.groupby(weight["source_itemid"].map({226512:"kg-native", 226531:"lb-converted"}))["weight_kg"].describe()

In [ ]:
weight_source_summary = (
    weight["source_itemid"].map({WEIGHT_ITEMID_KG: "kg-native",WEIGHT_ITEMID_LB: "lb-converted",WEIGHT_ITEMID_KG_DAILY:"kg-daily"})
    .value_counts().rename_axis("Weight source").reset_index(name="ICU stays"))
weight_source_summary["Percent"] = (weight_source_summary["ICU stays"] /len(weight) * 100).round(2)
print(weight_source_summary)
weight_source_summary.to_csv(OUTPUT / "weight_source_summary.csv",index=False)

In [ ]:
weight = weight[ [ "stay_id","weight_kg"]]#considering only required columns

In [ ]:
#calculates the time difference between outputevent measurement and the ICU intime(in hours)
outputevents["hours_from_icu"] = (outputevents["charttime"]-outputevents["intime"]).dt.total_seconds()/3600

#### Early AKI Assessment 0-24 Hours

In [ ]:
uo_0_24 = get_time_window(outputevents, 0, 24)

In [ ]:
uo_0_24 = uo_0_24.merge(cohort[["stay_id"]],on="stay_id",how="inner")

In [ ]:
# Aggregate urine output to hourly bins first
hourly_uo_0_24_raw=aggregate_hourly_output(uo_0_24)

print("0-24h WINDOW")
print("Urine-available stays (independent of weight):",hourly_uo_0_24_raw["stay_id"].nunique())
print("Weight-available stays (independent of urine):",weight["stay_id"].nunique())
print("Both available:", len(set(hourly_uo_0_24_raw["stay_id"]) & set(weight["stay_id"])))

In [ ]:
#early urine elapsed time labels
early_urine_labels_et = generate_urine_labels_elapsed_time(outputevents[outputevents["stay_id"].isin(cohort["stay_id"])],weight, window_start=0, window_end=24)

In [ ]:
early_urine_labels = early_urine_labels_et.rename(columns={"urine_labels_et": "urine_labels"})[["stay_id", "urine_labels"]]

print("Early urine-output evaluable:", early_urine_labels_et["urine_evaluable_et"].sum())
print("Early urine-output AKI:", early_urine_labels_et["urine_labels_et"].sum())


### 5. Exclusion of Early AKI

In [ ]:
# combine early creatinine based and urine based AKI
early_kdigo = early_scr_labels.merge(early_urine_labels,on="stay_id",how="left")

early_kdigo["urine_labels"] = (early_kdigo["urine_labels"].fillna(False))
# Early KDIGO based on both urine and scr criteria
early_kdigo["early_AKI"] = (early_kdigo["aki_scr"] | early_kdigo["urine_labels"])

In [ ]:
eligible_cohort = cohort.merge(early_kdigo[["stay_id","early_AKI"]], on="stay_id",how="left")

print("Before removing early AKI:",eligible_cohort["stay_id"].nunique())
eligible_cohort["early_AKI"] = (eligible_cohort["early_AKI"].fillna(False))
#remove early AKI from eligible cohort
eligible_cohort = eligible_cohort[~eligible_cohort["early_AKI"]].copy()
print("After removing early AKI:", eligible_cohort["stay_id"].nunique())


### 6.AKI between 24-48 hours

#### KDIGO Creatinine criteria

In [ ]:
#only measurements during the 24-48 h outcome window
scr_24_48 = get_time_window(creatinine, 24, 48)

In [ ]:
scr_24_48 = scr_24_48.merge(eligible_cohort[["stay_id"]],on="stay_id",how="inner")

In [ ]:
print(scr_24_48["stay_id"].nunique())

In [ ]:
# satisfies KDIGO after the betweeen 24 and 48 hours.

# >=1.5x baseline must be reached during the 24-48 h outcome window
scr_15x_labels = kdigo_scr_15x(scr_24_48,baseline_scr)
# >=0.3 mg/dL rise, later measurement must be 24-48 h, earlier measurement may be anywhere in preceding 48 h
creatinine_outcome = creatinine.merge(eligible_cohort[["stay_id"]],on="stay_id",how="inner")

scr_03_labels = kdigo_scr_03( creatinine_outcome,outcome_start=24, outcome_end=48)
scr_labels = combine_scr_labels(eligible_cohort,scr_15x_labels,scr_03_labels)

### 7.Creatinine 24-hour Boundary Validation

In [ ]:
scr_labels["aki_scr"].value_counts()

In [ ]:
print("1.5x Baseline:",scr_labels["aki_1_5x"].sum())

print("0.3 mg/dL:",scr_labels["aki_0_3"].sum())

print("Serum Creatinine AKI:",scr_labels["aki_scr"].sum())

In [ ]:
print(eligible_cohort.shape)

In [ ]:
print("After early AKI exclusion:", eligible_cohort["stay_id"].nunique())

### 8.KDIGO Urine Criteria - 24-48 hours

In [ ]:
uo_24_48 = get_time_window(outputevents, 24,48)

In [ ]:
# keep only eligible ICU stays
uo_24_48 = uo_24_48.merge(eligible_cohort[["stay_id"]],on="stay_id",how="inner")

In [ ]:

# Elapsed-time labels
urine_labels_et = generate_urine_labels_elapsed_time(outputevents[outputevents["stay_id"].isin(eligible_cohort["stay_id"])], weight, window_start=24, window_end=48)
print("Elapsed-time evaluable stays:", urine_labels_et["urine_evaluable_et"].sum())
print("Elapsed-time UO-AKI positive:", urine_labels_et["urine_labels_et"].sum())


In [ ]:
# Aggregate urine output to hourly bins
hourly_uo_raw = aggregate_hourly_output(uo_24_48)

print("24-48h WINDOW")
print("Urine-available stays independent of weight:", hourly_uo_raw["stay_id"].nunique())
print("Weight-available stays independent of urine:", weight["stay_id"].nunique())
print("Both available:", len(set(hourly_uo_raw["stay_id"]) & set(weight["stay_id"])))

### 9. Urine Output Evaluability

In [ ]:

# weight available
weight_available = weight[weight["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique()
# Any hourly urine data
uo_available = hourly_uo_raw[hourly_uo_raw["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique()

# Number of observed hourly bins per stay
hour_counts = hourly_uo_raw.groupby("stay_id").size()
median_hours = hour_counts.median()
iqr_hours = (hour_counts.quantile(0.25),hour_counts.quantile(0.75))

urine_evaluable_ids = set(urine_labels_et.loc[urine_labels_et["urine_evaluable_et"], "stay_id"])

eligible_6h = len(urine_evaluable_ids)
print("Stays evaluable under elapsed-time method:", eligible_6h)


print("Weight available:", weight_available)
print("Any urine hours (ICU stays with >=1 observed hourly urine bin):",uo_available)
print("Median observed hourly urine bins:", median_hours)
print("IQR:", iqr_hours)
print("Elapsed >= 6 h:",eligible_6h)


In [ ]:

# Exclude stays with neither a usable serum baseline nor urine-output evaluability
no_scr = set(baseline_scr.loc[baseline_scr["baseline_scr"].isna(), "stay_id"])
exclude_ids = no_scr - urine_evaluable_ids
eligible_before_missing_exclusion = eligible_cohort["stay_id"].nunique()
print("Eligible cohort before exclusion:", eligible_cohort["stay_id"].nunique())
print("Missing both serum baseline and evaluable urine output:", eligible_cohort["stay_id"].isin(exclude_ids).sum())

eligible_cohort = eligible_cohort[~eligible_cohort["stay_id"].isin(exclude_ids)].copy()
print("Eligible cohort after exclusion:", eligible_cohort["stay_id"].nunique())

scr_labels = scr_labels[scr_labels["stay_id"].isin(eligible_cohort["stay_id"])].copy()

In [ ]:
# Recalculate baseline summary using the FINAL modeling cohort
baseline_scr_final = baseline_scr.merge(eligible_cohort[["stay_id"]],on="stay_id",how="inner")
baseline_scr_final["baseline_source"] = np.where(baseline_scr_final["pre_icu_baseline"].notna(),"Pre-ICU","First ICU")

pre_icu = baseline_scr_final["pre_icu_baseline"].notna().sum()
fallback = (baseline_scr_final["pre_icu_baseline"].isna()& baseline_scr_final["first_scr"].notna()).sum()
no_baseline = baseline_scr_final["baseline_scr"].isna().sum()

print(f"Final cohort: {len(baseline_scr_final):,}")
print(f"Pre-ICU baseline available: {pre_icu:,}")
print(f"Used first ICU creatinine as baseline: " f"{fallback:,} " f"({fallback / len(baseline_scr_final) * 100:.1f}%)")
print(f"No baseline creatinine available: "f"{no_baseline:,}")

In [ ]:
sensitivity_ids = set(baseline_scr_final.loc[baseline_scr_final["pre_icu_baseline"].notna(),"stay_id"])

print("Sensitivity cohort:", len(sensitivity_ids))

In [ ]:
final_n = eligible_cohort["stay_id"].nunique()
final_hourly_uo_check = hourly_uo_raw[hourly_uo_raw["stay_id"].isin(eligible_cohort["stay_id"])]

weight_available = weight[weight["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique()
final_hourly_uo_raw = hourly_uo_raw[hourly_uo_raw["stay_id"].isin(eligible_cohort["stay_id"])].copy()

uo_available = hourly_uo_raw[hourly_uo_raw["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique()
hour_counts = final_hourly_uo_raw.groupby("stay_id").size()

median_hours = hour_counts.median()
iqr_hours = (hour_counts.quantile(0.25),hour_counts.quantile(0.75))

final_urine_evaluable_ids = (set(urine_evaluable_ids)& set(eligible_cohort["stay_id"]))

eligible_6h = len(final_urine_evaluable_ids)
urine_evaluable_ids = final_urine_evaluable_ids

### 10.Mutually Exclusive Cohort Flow

In [ ]:
cohort_ids = set(cohort["stay_id"])
early_aki_ids = set(early_kdigo.loc[early_kdigo["early_AKI"], "stay_id"])
post_early_ids = cohort_ids - early_aki_ids
baseline_map = baseline_scr.set_index("stay_id")["baseline_scr"]
no_baseline_ids = {stay_id for stay_id in post_early_ids if pd.isna(baseline_map.get(stay_id))}
baseline_available_ids = post_early_ids - no_baseline_ids

# No serum baseline but urine output can rescue evaluability
baseline_missing_but_urine_evaluable = (no_baseline_ids & urine_evaluable_ids)
# These have neither serum baseline nor sufficient urine-output data
missing_both_ids = (no_baseline_ids - urine_evaluable_ids)
final_ids = post_early_ids - missing_both_ids
assert len(final_ids) == eligible_cohort["stay_id"].nunique()

In [ ]:
cohort_flow_corrected = pd.DataFrame([
    {
        "Step": "Initial eligible ICU cohort",
        "ICU Stays": len(cohort_ids),
        "Exclusion": 0
    },
    {
        "Step": "Exclude AKI during first 24 h",
        "ICU Stays": len(post_early_ids),
        "Exclusion": len(early_aki_ids)
    },
    {
        "Step": "No baseline creatinine but urine-output evaluable",
        "ICU Stays": len(baseline_missing_but_urine_evaluable),
        "Exclusion": 0
    },
    {
        "Step": "Neither serum creatinine nor urine output evaluable",
        "ICU Stays": len(missing_both_ids),
        "Exclusion": len(missing_both_ids)
    },
    {
        "Step": "Final modeling cohort",
        "ICU Stays": len(final_ids),
        "Exclusion": 0
    }
])

In [ ]:
cohort_flow_corrected.to_csv(OUTPUT/"cohort_flow_corrected.csv", index=False)

In [ ]:
urine_labels = urine_labels_et.rename(columns={"urine_labels_et": "urine_labels"})[["stay_id", "urine_labels"]]

In [ ]:
urine_positive_ids = set(urine_labels_et.loc[urine_labels_et["urine_labels_et"], "stay_id"])
urine_evaluable_et_ids = set(urine_labels_et.loc[urine_labels_et["urine_evaluable_et"], "stay_id"])
print("Urine-output AKI cases (elapsed-time):", len(urine_positive_ids))
print("Of those, also in final urine_evaluable_ids (post-cohort-filtering):", len(urine_positive_ids & set(urine_evaluable_ids)))

In [ ]:
urine_labels["urine_labels"].value_counts()

In [ ]:
print("Eligible cohort:", eligible_cohort["stay_id"].nunique())
print("ICU Stays with urine data:",hourly_uo_raw["stay_id"].nunique())

### 10. Urine Output Evaluability Summary

In [ ]:
final_n = eligible_cohort["stay_id"].nunique()

urine_eval = pd.DataFrame({
    "Metric":[
        "Final modeling cohort",
        "Usable body weight",
        ">=1 observed hourly urine bin",
        "Urine evaluable (elapsed ≥ 6 h)",
        "Median observed hourly bins",
        "IQR observed hourly bins"
    ],
    "Count":[
        final_n,
        weight_available,
        uo_available,
        eligible_6h,
        median_hours,
        np.nan
    ],
    "Percent":[
        100.0,
        round(weight_available/final_n*100,1),
        round(uo_available/final_n*100,1),
        round(eligible_6h/final_n*100,1),
        np.nan,
        np.nan
    ],
    "Notes":[
        "",
        "",
        "",
        "",
        "",
        f"{iqr_hours[0]}–{iqr_hours[1]}"
    ]
})

urine_eval

In [ ]:
urine_eval.to_csv(OUTPUT/"urine_evaluability_summary.csv",index=False)

### 11. Final KDIGO AKI Label Generation

In [ ]:
# Merging AKI labels with Scr labels
aki_labels = scr_labels.merge(urine_labels_et, on="stay_id", how="left")
aki_labels["urine_evaluable"] = aki_labels["urine_evaluable_et"].fillna(False)
aki_labels["urine_labels"] = aki_labels["urine_labels_et"].fillna(False)
aki_labels["AKI"] = aki_labels["aki_scr"] | aki_labels["urine_labels"]

### 12. CRRT Auto-Staging

In [ ]:

import duckdb

con = duckdb.connect("mimic-iv.duckdb")
con.execute(f"COPY mimiciv_derived.crrt TO '{DATA / 'derived' / 'crrt.csv'}' (HEADER, DELIMITER ',')")
con.close()
crrt_csv = DATA / "derived" / "crrt.csv"
if not crrt_csv.exists():
    con = duckdb.connect("mimic-iv.duckdb")
    con.execute(f"COPY mimiciv_derived.crrt TO '{crrt_csv}' (HEADER, DELIMITER ',')")
    con.close()
crrt = pd.read_csv(crrt_csv, parse_dates=["charttime"])

crrt_ref_full = crrt.merge(cohort[["stay_id","intime"]], on="stay_id", how="inner")
crrt_ref_full["hours_from_icu"] = (crrt_ref_full["charttime"] - crrt_ref_full["intime"]).dt.total_seconds() / 3600

crrt_active_24_48 = set( crrt_ref_full[(crrt_ref_full["hours_from_icu"] >= 24) & (crrt_ref_full["hours_from_icu"] < 48)]["stay_id"])
aki_labels["on_crrt"] = aki_labels["stay_id"].isin(crrt_active_24_48)
print("Stays on CRRT during 24-48h window:", aki_labels["on_crrt"].sum())
print("Of those, previously AKI-negative before CRRT rule applied:",(aki_labels["on_crrt"] & ~(aki_labels["aki_scr"] | aki_labels["urine_labels"])).sum())

# CRRT patients are AKI-positive by definition for CRRT staging
aki_labels["AKI"] = aki_labels["AKI"] | aki_labels["on_crrt"]
print("Final AKI positive count after CRRT rule:", aki_labels["AKI"].sum())

In [ ]:

aki_labels["label_category"] = np.select(
    [
        aki_labels["AKI"] == True,
        (aki_labels["AKI"] == False) & (aki_labels["aki_scr"] == False) & (aki_labels["urine_evaluable"] == False),
        (aki_labels["AKI"] == False)
    ],
    ["AKI_positive", "unevaluable_negative", "AKI_negative"],
    default="AKI_negative"
)
print(aki_labels["label_category"].value_counts())

#### 13. Comparison with mimiciv_derived official KDIGO reference tables

In [ ]:
kdigo_stages_ref = pd.read_csv(DERIVED_DIR / "kdigo_stages.csv", parse_dates=["charttime"])

ref = kdigo_stages_ref.merge(cohort[["stay_id","intime"]], on="stay_id", how="inner")
ref["hours_from_icu"] = (ref["charttime"] - ref["intime"]).dt.total_seconds() / 3600
ref_24_48 = ref[(ref["hours_from_icu"] >= 24) & (ref["hours_from_icu"] < 48)]
ref_label = (ref_24_48.groupby("stay_id")["aki_stage"].max().reset_index().assign(AKI_ref=lambda d: d["aki_stage"] > 0))


In [ ]:
ref_label_creat = (ref_24_48.groupby("stay_id")["aki_stage_creat"].max().reset_index().assign(AKI_ref_creat=lambda d: d["aki_stage_creat"] > 0))
ref_label_uo = (ref_24_48.groupby("stay_id")["aki_stage_uo"].max().reset_index().assign(AKI_ref_uo=lambda d: d["aki_stage_uo"] > 0))

compare = aki_labels.merge(ref_label[["stay_id","AKI_ref"]], on="stay_id", how="inner")
print("Stays compared (combined AKI):", len(compare))
print("Agreement:", round((compare["AKI"] == compare["AKI_ref"]).mean(), 4))
print(pd.crosstab(compare["AKI"], compare["AKI_ref"], margins=True))

In [ ]:
from sklearn.metrics import cohen_kappa_score
kappa_combined = cohen_kappa_score(compare["AKI"], compare["AKI_ref"])
print("Cohen's kappa (combined):", round(kappa_combined, 4))

compare_creat = aki_labels.merge(ref_label_creat[["stay_id","AKI_ref_creat"]], on="stay_id", how="inner")
print("Creatinine-only agreement:", round((compare_creat["aki_scr"] == compare_creat["AKI_ref_creat"]).mean(), 4))
print(pd.crosstab(compare_creat["aki_scr"], compare_creat["AKI_ref_creat"], margins=True))

compare_uo = aki_labels.merge(ref_label_uo[["stay_id","AKI_ref_uo"]], on="stay_id", how="inner")
print("Urine-output-only agreement:", round((compare_uo["urine_labels"] == compare_uo["AKI_ref_uo"]).mean(), 4))
print(pd.crosstab(compare_uo["urine_labels"], compare_uo["AKI_ref_uo"], margins=True))


In [ ]:
from sklearn.metrics import confusion_matrix

def full_stats(y_true, y_pred, label):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens =tp / (tp + fn) if (tp+fn) else float('nan')
    spec  =tn / (tn + fp) if (tn+fp) else float('nan')
    ppv =tp / (tp + fp) if (tp+fp) else float('nan')
    npv =tn / (tn + fn) if (tn+fn) else float('nan')
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f" {label}")
    print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
    print(f"Sensitivity: {sens:.4f}  Specificity: {spec:.4f}")
    print(f"PPV: {ppv:.4f}  NPV: {npv:.4f}  Kappa: {kappa:.4f}")
    print()
    return {"label": label, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "sensitivity": sens, "specificity": spec, "ppv": ppv, "npv": npv, "kappa": kappa}


In [ ]:
# confusion_matrix unpacking order 
print("Raw confusion matrix (creatinine):")
print(confusion_matrix(compare_creat["AKI_ref_creat"], compare_creat["aki_scr"]))
print("Expected layout: [[TN, FP], [FN, TP]] when labels sorted [False, True]")


In [ ]:
stats_creat = full_stats(compare_creat["AKI_ref_creat"], compare_creat["aki_scr"], "Creatinine")
stats_uo = full_stats(compare_uo["AKI_ref_uo"], compare_uo["urine_labels"], "Urine Output")

full_stats_df = pd.DataFrame([stats_creat, stats_uo])
full_stats_df.to_csv(OUTPUT / "full_validation_stats.csv", index=False)


##### Comparison with mimiciv_derived.kdigo_creatinine and kdigo_uo (raw tables)

In [ ]:
kdigo_creatinine_ref = pd.read_csv(DERIVED_DIR / "kdigo_creatinine.csv", parse_dates=["charttime"])
kdigo_uo_ref = pd.read_csv(DERIVED_DIR / "kdigo_uo.csv", parse_dates=["charttime"])

kdigo_creatinine_ref = kdigo_creatinine_ref.merge(cohort[["stay_id","intime"]], on="stay_id", how="inner")
kdigo_creatinine_ref["hours_from_icu"] = (kdigo_creatinine_ref["charttime"] - kdigo_creatinine_ref["intime"]).dt.total_seconds() / 3600
kdigo_creatinine_24_48 = kdigo_creatinine_ref[ (kdigo_creatinine_ref["hours_from_icu"] >= 24) & (kdigo_creatinine_ref["hours_from_icu"] < 48)].copy()


In [ ]:
kdigo_uo_ref = kdigo_uo_ref.merge(cohort[["stay_id","intime"]], on="stay_id", how="inner")
kdigo_uo_ref["hours_from_icu"] = (kdigo_uo_ref["charttime"] - kdigo_uo_ref["intime"]).dt.total_seconds() / 3600
kdigo_uo_24_48_ref = kdigo_uo_ref[(kdigo_uo_ref["hours_from_icu"] >= 24) & (kdigo_uo_ref["hours_from_icu"] < 48)].copy()

print("kdigo_creatinine.csv rows in 24-48h window:", len(kdigo_creatinine_24_48))
print("kdigo_uo.csv rows in 24-48h window:", len(kdigo_uo_24_48_ref))

In [ ]:
kdigo_creatinine_24_48["aki_creat_raw"] = (
(kdigo_creatinine_24_48["creat"] - kdigo_creatinine_24_48["creat_low_past_48hr"] >= 0.3) |(kdigo_creatinine_24_48["creat"] >= 1.5 * kdigo_creatinine_24_48["creat_low_past_7day"]))
ref_creat_raw_label = (kdigo_creatinine_24_48.groupby("stay_id")["aki_creat_raw"].max().reset_index().rename(columns={"aki_creat_raw": "AKI_ref_creat_raw"}))


In [ ]:

kdigo_uo_24_48_ref["aki_uo_raw"] = kdigo_uo_24_48_ref["uo_rt_6hr"] < 0.5
ref_uo_raw_label = (
    kdigo_uo_24_48_ref.groupby("stay_id")["aki_uo_raw"].max().reset_index().rename(columns={"aki_uo_raw": "AKI_ref_uo_raw"}))

compare_creat_raw = aki_labels.merge(ref_creat_raw_label[["stay_id","AKI_ref_creat_raw"]], on="stay_id", how="inner")
kappa_creat_raw = cohen_kappa_score(compare_creat_raw["aki_scr"], compare_creat_raw["AKI_ref_creat_raw"])
print("Creatinine (raw kdigo_creatinine.csv) agreement:",round((compare_creat_raw["aki_scr"] == compare_creat_raw["AKI_ref_creat_raw"]).mean(), 4))
print("Creatinine (raw) kappa:", round(kappa_creat_raw, 4))


In [ ]:

compare_uo_raw = aki_labels.merge(ref_uo_raw_label[["stay_id","AKI_ref_uo_raw"]], on="stay_id", how="inner")
kappa_uo_raw = cohen_kappa_score(compare_uo_raw["urine_labels"], compare_uo_raw["AKI_ref_uo_raw"])
print("Urine (raw kdigo_uo.csv) agreement:",round((compare_uo_raw["urine_labels"] == compare_uo_raw["AKI_ref_uo_raw"]).mean(), 4))
print("Urine (raw) kappa:", round(kappa_uo_raw, 4))


In [ ]:

stages_vs_raw_creat = ref_label_creat.merge(ref_creat_raw_label, on="stay_id", how="inner")
print("kdigo_stages-derived vs raw kdigo_creatinine.csv agreement:",
      round((stages_vs_raw_creat["AKI_ref_creat"] == stages_vs_raw_creat["AKI_ref_creat_raw"]).mean(), 4))

stages_vs_raw_uo = ref_label_uo.merge(ref_uo_raw_label, on="stay_id", how="inner")
print("kdigo_stages-derived vs raw kdigo_uo.csv agreement:",
      round((stages_vs_raw_uo["AKI_ref_uo"] == stages_vs_raw_uo["AKI_ref_uo_raw"]).mean(), 4))


In [ ]:
mimic_derived_comparison_raw = pd.DataFrame({
    "Comparison": ["Creatinine (raw kdigo_creatinine.csv)", "Urine (raw kdigo_uo.csv)"],
    "Agreement": [
        round((compare_creat_raw["aki_scr"] == compare_creat_raw["AKI_ref_creat_raw"]).mean(), 4),
        round((compare_uo_raw["urine_labels"] == compare_uo_raw["AKI_ref_uo_raw"]).mean(), 4),
    ],
    "Cohens_kappa": [round(kappa_creat_raw, 4), round(kappa_uo_raw, 4)],
    "N_stays_compared": [len(compare_creat_raw), len(compare_uo_raw)]
})
mimic_derived_comparison_raw.to_csv(OUTPUT / "mimic_derived_comparison_raw_tables.csv", index=False)
mimic_derived_comparison_raw


In [ ]:

compare["has_crrt"] = compare["stay_id"].isin(crrt_active_24_48)
print("CRRT stays in comparison population:", compare["has_crrt"].sum())
print("Agreement excluding CRRT stays:", round((compare.loc[~compare["has_crrt"],"AKI"]==compare.loc[~compare["has_crrt"],"AKI_ref"]).mean(),4))
print("Agreement among CRRT stays only:", round((compare.loc[compare["has_crrt"],"AKI"]==compare.loc[compare["has_crrt"],"AKI_ref"]).mean(),4))


In [ ]:

mimic_derived_comparison = pd.DataFrame({
    "Comparison": ["Combined AKI", "Creatinine-only", "Urine-output-only"],
    "Agreement": [
        round((compare["AKI"] == compare["AKI_ref"]).mean(), 4),
        round((compare_creat["aki_scr"] == compare_creat["AKI_ref_creat"]).mean(), 4),
        round((compare_uo["urine_labels"] == compare_uo["AKI_ref_uo"]).mean(), 4),
    ],
    "Cohens_kappa": [
        round(kappa_combined, 4),
        round(cohen_kappa_score(compare_creat["aki_scr"], compare_creat["AKI_ref_creat"]), 4),
        round(cohen_kappa_score(compare_uo["urine_labels"], compare_uo["AKI_ref_uo"]), 4),
    ],
    "N_stays_compared": [len(compare), len(compare_creat), len(compare_uo)]
})
mimic_derived_comparison.to_csv(OUTPUT / "mimic_derived_comparison.csv", index=False)
mimic_derived_comparison


In [ ]:
discrepant = compare[compare["AKI"] != compare["AKI_ref"]]
discrepant.to_csv(OUTPUT / "discrepant_vs_mimic_derived.csv", index=False)
print(f"Discrepant stays saved: {len(discrepant):,}")

In [ ]:
ref_uo_evaluable_ids = set(ref_24_48.loc[ref_24_48["uo_rt_6hr"].notna(), "stay_id"])

pipeline_evaluable_ids = urine_evaluable_ids
print("Pipeline UOevaluable stays:", len(pipeline_evaluable_ids))
print("Reference UOevaluable stays:", len(ref_uo_evaluable_ids))
print("Overlap:", len(pipeline_evaluable_ids & ref_uo_evaluable_ids))
print("Reference evaluable but pipeline is NOT:", len(ref_uo_evaluable_ids - pipeline_evaluable_ids))
print("Pipeline evaluable but reference is NOT:", len(pipeline_evaluable_ids - ref_uo_evaluable_ids))

### 14. Sensitivity Analysis

#### Pre-ICU Baseline Cohort sensitivity Analysis

In [ ]:
aki_sensitivity = aki_labels[aki_labels["stay_id"].isin(sensitivity_ids)].copy()

sensitivity_summary = pd.DataFrame({
    "Metric": [
        "Sensitivity cohort",
        "AKI Positive",
        "AKI Negative",
        "AKI prevalence (%)"
    ],
    "Value": [
        len(aki_sensitivity),
        int(aki_sensitivity["AKI"].sum()),
        int((~aki_sensitivity["AKI"]).sum()),
        round( aki_sensitivity["AKI"].mean()*100,2)
    ]
})

print(sensitivity_summary)
sensitivity_summary.to_csv(Sensitivity_OUTPUT/"sensitivity_preicu_baseline.csv",index=False)

In [ ]:
eligible_cohort_sensitivity = eligible_cohort[eligible_cohort["stay_id"].isin(sensitivity_ids)].copy()
eligible_cohort_sensitivity.to_parquet(Sensitivity_OUTPUT / "cohort_sensitivity.parquet",index=False)

labels_sensitivity = aki_labels[aki_labels["stay_id"].isin(sensitivity_ids)].copy()
labels_sensitivity.to_parquet(Sensitivity_OUTPUT / "labels_sensitivity.parquet",index=False)

#### Urine-Output-Evaluable Cohort Sensitivity Analysis

In [ ]:
urine_sensitivity_ids = urine_evaluable_ids

aki_urine_sensitivity = aki_labels[aki_labels["stay_id"].isin(urine_sensitivity_ids)].copy()

urine_sensitivity_summary = pd.DataFrame({
    "Metric": [
        "Urine-evaluable sensitivity cohort",
        "AKI Positive",
        "AKI Negative",
        "AKI prevalence (%)"
    ],
    "Value": [
        len(aki_urine_sensitivity),
        int(aki_urine_sensitivity["AKI"].sum()),
        int((~aki_urine_sensitivity["AKI"]).sum()),
        round(aki_urine_sensitivity["AKI"].mean()*100, 2)
    ]
})

print(urine_sensitivity_summary)
urine_sensitivity_summary.to_csv(Sensitivity_OUTPUT / "sensitivity_urine_evaluable.csv", index=False)


eligible_cohort_urine_sensitivity = eligible_cohort[eligible_cohort["stay_id"].isin(urine_sensitivity_ids)].copy()
eligible_cohort_urine_sensitivity.to_parquet(Sensitivity_OUTPUT / "cohort_sensitivity_urine_evaluable.parquet", index=False)

labels_urine_sensitivity = aki_labels[aki_labels["stay_id"].isin(urine_sensitivity_ids)].copy()
labels_urine_sensitivity.to_parquet(Sensitivity_OUTPUT / "labels_sensitivity_urine_evaluable.parquet", index=False)


print(f"Full cohort AKI prevalence: {aki_labels['AKI'].mean()*100:.2f}% (n={len(aki_labels):,})")
print(f"Urine-evaluable-only AKI prevalence: {aki_urine_sensitivity['AKI'].mean()*100:.2f}% (n={len(aki_urine_sensitivity):,})")

### 15. AKI Outcome Prevalence and Label Summary

In [ ]:
print("Early Serum AKI:")
print(early_kdigo["aki_scr"].value_counts())

print("\nEarly Urine AKI:")
print(early_kdigo["urine_labels"].value_counts())

print("\nEarly KDIGO AKI:")
print(early_kdigo["early_AKI"].value_counts())

print( "\nEarly AKI prevalence:",round(early_kdigo["early_AKI"].mean() * 100, 2),"%")

In [ ]:
print("Serum AKI (24-48 h):")
print(aki_labels["aki_scr"].value_counts())

print("\nUrine AKI (24-48 h):")
print(aki_labels["urine_labels"].value_counts())

print("\nFinal KDIGO AKI (24-48 h):")
print(aki_labels["AKI"].value_counts())

print("\nAKI prevalence:",round(aki_labels["AKI"].mean()*100, 2),"%")

In [ ]:
aki_source = aki_labels.copy()

aki_source["AKI_source"] = np.select(
    [
        aki_source["on_crrt"],
        aki_source["aki_scr"] & aki_source["urine_labels"],
        aki_source["aki_scr"] & ~aki_source["urine_labels"],
        ~aki_source["aki_scr"] & aki_source["urine_labels"]
    ],
    ["CRRT", "Both criteria", "Creatinine only", "Urine output only"],
    default="Neither"
)

source_summary = (aki_source["AKI_source"].value_counts().rename_axis("AKI detection source").reset_index(name="ICU stays"))
source_summary["Percent of final cohort"] = (source_summary["ICU stays"] /len(aki_source) * 100).round(2)

source_summary

In [ ]:
source_summary.to_csv(OUTPUT / "aki_detection_source_summary.csv",index=False)

In [ ]:
overlap = aki_labels.copy()
overlap["urine_evaluable"] = overlap["stay_id"].isin(urine_evaluable_ids)
pd.crosstab(overlap["aki_scr"], overlap["urine_labels"],margins=True)


In [ ]:
pd.crosstab(overlap["urine_evaluable"],overlap["AKI"],margins=True)

### 16. Baseline Creatinine and Urine-Output Rescue

In [ ]:
baseline_summary = pd.DataFrame({
    "Metric": [
        "Final modeling cohort",
        "Pre-ICU baseline available",
        "Used first ICU creatinine",
        "No baseline creatinine available"
    ],
    "ICU Stays": [
        len(baseline_scr_final),
        pre_icu,
        fallback,
        no_baseline
    ]
})

baseline_summary["Percent of final cohort"] = (
    baseline_summary["ICU Stays"] /
    len(baseline_scr_final) * 100
).round(2)

baseline_summary



In [ ]:
print(f"Baseline-creatinine summary denominator: "f"{len(baseline_scr_final):,} ICU stays ")

In [ ]:
serum_evaluable_ids = set(baseline_scr_final.loc[baseline_scr_final["baseline_scr"].notna(),"stay_id"])


both_evaluable = serum_evaluable_ids & urine_evaluable_ids
neither_evaluable = (set(eligible_cohort["stay_id"])-(serum_evaluable_ids | urine_evaluable_ids))

overlap_summary = pd.DataFrame({
    "Metric":[
        "Final modeling cohort",
        "1.5× serum criterion baseline available",
        "Urine evaluable",
        "Both evaluable",
        "Neither evaluable"
    ],
    "Count":[
        eligible_cohort["stay_id"].nunique(),
        len(serum_evaluable_ids),
        len(urine_evaluable_ids),
        len(both_evaluable),
        len(neither_evaluable)
    ],
    "Percent":[
        100.0,
        round(len(serum_evaluable_ids)/final_n*100,1),
        round(len(urine_evaluable_ids)/final_n*100,1),
        round(len(both_evaluable)/final_n*100,1),
        round(len(neither_evaluable)/final_n*100,1)]
})
overlap_summary

In [ ]:
baseline_scr_final[["stay_id","baseline_scr","baseline_source" ]].to_parquet(OUTPUT / "baseline_scr.parquet",index=False)

In [ ]:
overlap_summary.to_csv(OUTPUT/"overlap_summary.csv",index=False)

In [ ]:

# Urine-output evaluability versus KDIGO urine AKI

urine_eval_table = pd.DataFrame({
    "Metric": [
        "Final modeling cohort",
        "Urine evaluable (elapsed ≥ 6 h)",
        "Urine non-evaluable (elapsed < 6 h)",
        "Urine-output AKI positive",
        "Urine-output AKI negative among evaluable"
    ],
    "Count": [
        final_n,
        len(urine_evaluable_ids),
        final_n - len(urine_evaluable_ids),
        int(urine_labels["urine_labels"].sum()),
        len(urine_evaluable_ids) - int(urine_labels["urine_labels"].sum())
    ]
})
urine_eval_table["Percent (%)"] = ( urine_eval_table["Count"] / final_n * 100).round(1)

urine_eval_table

In [ ]:
urine_eval_table.to_csv(OUTPUT / "urine_evaluability_vs_labels.csv",index=False)

In [ ]:
print(f"{len(urine_evaluable_ids):,} of {final_n:,} ICU stays " f"({len(urine_evaluable_ids)/final_n*100:.1f}%) " "were evaluable for the KDIGO urine-output criterion.")
print( f"{int(urine_labels['urine_labels'].sum()):,} ""urine-output AKI cases were identified among evaluable ICU stays.")

In [ ]:
labels = aki_labels[["stay_id","aki_0_3", "aki_1_5x","urine_labels","aki_scr","urine_evaluable","AKI"]].copy()
labels.to_parquet(OUTPUT / "labels.parquet",index=False)

In [ ]:
positive_aki = int(aki_labels["AKI"].sum())
negative_aki = int((~aki_labels["AKI"]).sum())

label_summary = pd.DataFrame({
    "Metric": [
        "Eligible Modeling Cohort",
        "Serum Creatinine AKI",
        "Urine Output AKI",
        "Final KDIGO AKI (Positive)",
        "Final Non-AKI (Negative)",
        "AKI Prevalence (%)"
    ],
    "Value": [
        eligible_cohort["stay_id"].nunique(),
        int(aki_labels["aki_scr"].sum()),
        int(aki_labels["urine_labels"].sum()),
        positive_aki,
        negative_aki,
        round(100 * aki_labels["AKI"].mean(), 2)
    ]
})

label_summary


In [ ]:
label_summary.to_csv(OUTPUT / "label_summary.csv",index=False)

In [ ]:
print("No serum baseline:", len(no_scr))
print("Urine evaluable:", len(urine_evaluable_ids))
print("Excluded:", len(exclude_ids))
print("Eligible before exclusion:",eligible_before_missing_exclusion)
print("Eligible after exclusion:",eligible_cohort["stay_id"].nunique())

In [ ]:
print( baseline_scr["baseline_scr"].isna().sum() )
print(baseline_scr_final["baseline_scr"].isna().sum())

### 17. Cohort Flow and Final Class Distribution

In [ ]:
class_summary = pd.DataFrame({
    "Metric": [
        "Final modeling cohort",
        "AKI Positive",
        "AKI Negative",
        "AKI Prevalence (%)"
    ],
    "Value": [
        eligible_cohort["stay_id"].nunique(),
        aki_labels["AKI"].sum(),
        (~aki_labels["AKI"]).sum(),
        round(aki_labels["AKI"].mean() * 100, 2)
    ]
})

class_summary


In [ ]:
class_summary.to_csv(OUTPUT / "class_summary.csv",index=False)

#### Label and Cohort Validations

In [ ]:
# 1. One label per ICU stay
assert aki_labels["stay_id"].is_unique

# 2. Every eligible stay has a label
assert (aki_labels["stay_id"].nunique() ==eligible_cohort["stay_id"].nunique())

# 3. Final counts
positive = aki_labels["AKI"].sum()
negative = (~aki_labels["AKI"]).sum()

print("Positive:", positive)
print("Negative:", negative)
print("Total:", positive + negative)

assert positive + negative == eligible_cohort["stay_id"].nunique()

In [ ]:
print("Final modeling cohort:", eligible_cohort["stay_id"].nunique())
print("Baseline available:",baseline_scr[baseline_scr["stay_id"].isin(eligible_cohort["stay_id"])]["baseline_scr"].notna().sum())

print("Missing baseline:",baseline_scr[baseline_scr["stay_id"].isin(eligible_cohort["stay_id"])]["baseline_scr"].isna().sum())

print("Weight available:", weight_available)
print("Any urine observations:", uo_available)
print(">=6 hourly bins:", eligible_6h)
print("Median hours:", median_hours)
print("IQR:", iqr_hours)

pd.crosstab(
    overlap["aki_scr"],
    overlap["urine_labels"],
    margins=True
)
pd.crosstab(
    aki_labels["urine_evaluable"],
    aki_labels["urine_labels"]
)

pd.crosstab(
    overlap["urine_evaluable"],
    overlap["AKI"],
    margins=True
)

print(label_summary)

In [ ]:
final_n = aki_labels["stay_id"].nunique()

positive = int(aki_labels["AKI"].sum())
negative = int((~aki_labels["AKI"]).sum())

print("KDIGO AKI COHORT")
print(f"Initial cohort:{len(cohort_ids):,}")
print(f"Early AKI excluded:{len(early_aki_ids):,}")
print(f"Post-early-AKI cohort: {len(post_early_ids):,}")
print(f"Missing baseline creatinine:{len(no_baseline_ids):,}")
print(f"Urine-output rescue: {len(baseline_missing_but_urine_evaluable):,}")
print(f"Excluded for missing both:{len(missing_both_ids):,}")
print(f"Final modeling cohort:{final_n:,}")

print()
print(f"AKI positive:{positive:,}")
print(f"AKI negative:{negative:,}")
print(f"AKI prevalence:{positive / final_n * 100:.2f}%")

print()
print("AKI detection sources")
print(f"Creatinine only:{int(((aki_labels.aki_scr) & (~aki_labels.urine_labels)).sum()):,}")
print(f"Urine output only:{int(((~aki_labels.aki_scr) & (aki_labels.urine_labels)).sum()):,}")
print(f"Both criteria:{int((aki_labels.aki_scr & aki_labels.urine_labels).sum()):,}")

    



In [ ]:
print("Consistency check for weight_available:", 
      weight[weight["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique())
print("Consistency check for uo_available:", 
      hourly_uo_raw[hourly_uo_raw["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique())

In [ ]:
print(cohort_flow_corrected)
print(baseline_summary)
print(overlap_summary)

In [ ]:
#  how often 227488 and 227489 appear at the SAME charttime for the same stay, vs. appearing unpaired, across the whole 24-48h outcome window
irrigant_events = uo_24_48[uo_24_48["itemid"].isin([227488, 227489])].copy()

paired_check = irrigant_events.groupby(["stay_id", "charttime"])["itemid"].apply(
    lambda x: set(x) == {227488, 227489}
)

print("Total irrigant-item timestamps:", len(paired_check))
print("Timestamps with BOTH items paired:", paired_check.sum())
print("Timestamps with only ONE item present:", (~paired_check).sum())
print("Fraction unpaired:", round((~paired_check).mean(), 4))

# How many distinct stays are affected at all?
stays_with_irrigant = irrigant_events["stay_id"].nunique()
print("Distinct stays using irrigant items (227488/227489) in 24-48h window:", stays_with_irrigant)

In [ ]:

irrigant_paired_status = irrigant_events.groupby(["stay_id", "charttime"])["itemid"].apply(
    lambda x: set(x) == {227488, 227489}
).reset_index(name="is_paired")

unpaired_stays = set(irrigant_paired_status.loc[~irrigant_paired_status["is_paired"], "stay_id"])
print(f"Stays with at least one unpaired irrigant timestamp: {len(unpaired_stays)}")
affected_check = aki_labels[aki_labels["stay_id"].isin(unpaired_stays)][
    ["stay_id", "urine_evaluable", "urine_labels", "AKI"]
]
print(affected_check.to_string(index=False))
print(f"\nOf these, urine-output AKI-positive: {affected_check['urine_labels'].sum()}")

In [ ]:
assert aki_labels["stay_id"].is_unique
assert (aki_labels["stay_id"].nunique() == eligible_cohort["stay_id"].nunique())
positive = aki_labels["AKI"].sum()
negative = (~aki_labels["AKI"]).sum()
assert positive + negative == eligible_cohort["stay_id"].nunique()
print("All assertions passed.")

In [ ]:
aki_labels["short_los"] = aki_labels["stay_id"].isin(short_stay_ids)
aki_labels["has_esrd"] = aki_labels["stay_id"].isin(esrd_dialysis_stay_ids)

stratified_summary = pd.DataFrame({
    "Subgroup": ["Full cohort", "LOS < 48h", "LOS >= 48h", "ESRD or chronic dialysis", "No ESRD/dialysis"],
    "N": [
        len(aki_labels),
        aki_labels["short_los"].sum(),
        (~aki_labels["short_los"]).sum(),
        aki_labels["has_esrd"].sum(),
        (~aki_labels["has_esrd"]).sum()
    ],
    "Urine_evaluable_pct": [
        aki_labels["urine_evaluable"].mean()*100,
        aki_labels.loc[aki_labels["short_los"],"urine_evaluable"].mean()*100,
        aki_labels.loc[~aki_labels["short_los"],"urine_evaluable"].mean()*100,
        aki_labels.loc[aki_labels["has_esrd"],"urine_evaluable"].mean()*100,
        aki_labels.loc[~aki_labels["has_esrd"],"urine_evaluable"].mean()*100,
    ],
    "AKI_prevalence_pct": [
        aki_labels["AKI"].mean()*100,
        aki_labels.loc[aki_labels["short_los"],"AKI"].mean()*100,
        aki_labels.loc[~aki_labels["short_los"],"AKI"].mean()*100,
        aki_labels.loc[aki_labels["has_esrd"],"AKI"].mean()*100,
        aki_labels.loc[~aki_labels["has_esrd"],"AKI"].mean()*100,
    ]
})
print(stratified_summary)
stratified_summary.to_csv(OUTPUT / "stratified_subgroup_summary.csv", index=False)

In [ ]:
# Creatinine
creatinine_outtime_check = creatinine.merge(cohort[["stay_id","outtime"]], on="stay_id", how="left")
creatinine_outtime_check["after_outtime"] = creatinine_outtime_check["charttime"] > creatinine_outtime_check["outtime"]

leaked_creatinine = creatinine_outtime_check[
    (creatinine_outtime_check["hours_from_icu"] >= 24) &
    (creatinine_outtime_check["hours_from_icu"] < 48) &
    (creatinine_outtime_check["after_outtime"])
]
print("Creatinine events after outtime, still counted in 24-48h window:", len(leaked_creatinine))
print("Distinct stays affected:", leaked_creatinine["stay_id"].nunique())

# Urine output
outputevents_outtime_check = outputevents.merge(cohort[["stay_id","outtime"]], on="stay_id", how="left")
outputevents_outtime_check["after_outtime"] = outputevents_outtime_check["charttime"] > outputevents_outtime_check["outtime"]

leaked_urine = outputevents_outtime_check[
    (outputevents_outtime_check["hours_from_icu"] >= 24) &
    (outputevents_outtime_check["hours_from_icu"] < 48) &
    (outputevents_outtime_check["after_outtime"])
]
print("Urine output events after outtime, still counted in 24-48h window:", len(leaked_urine))
print("Distinct stays affected:", leaked_urine["stay_id"].nunique())

In [ ]:
# Identify stays where the 24-48h outcome window includes any creatinine measurement
# charted after the stay's recorded ICU outtime 
post_outtime_creat_stay_ids = set(leaked_creatinine["stay_id"].unique())
post_outtime_urine_stay_ids = set(leaked_urine["stay_id"].unique())
post_outtime_any_stay_ids = post_outtime_creat_stay_ids | post_outtime_urine_stay_ids

aki_labels["post_outtime_data"] = aki_labels["stay_id"].isin(post_outtime_any_stay_ids)

print(f"Final cohort stays with post-outtime data in 24-48h window: {aki_labels['post_outtime_data'].sum()} "
      f"({aki_labels['post_outtime_data'].mean()*100:.1f}%)")

# Quick prevalence check, same pattern as ESRD/LOS
print(f"AKI prevalence, post-outtime-data group: {aki_labels.loc[aki_labels['post_outtime_data'],'AKI'].mean()*100:.2f}% "
      f"(n={aki_labels['post_outtime_data'].sum()})")
print(f"AKI prevalence, everyone else: {aki_labels.loc[~aki_labels['post_outtime_data'],'AKI'].mean()*100:.2f}%")

In [ ]:
print("Final cohort stays with LOS < 48h:", eligible_cohort["stay_id"].isin(short_stay_ids).sum())
print(f"As % of final cohort: {eligible_cohort['stay_id'].isin(short_stay_ids).sum() / eligible_cohort['stay_id'].nunique() * 100:.1f}%")

In [ ]:
n_flag = eligible_cohort["stay_id"].isin(esrd_dialysis_stay_ids).sum()
print("Final cohort stays with ESRD or chronic-dialysis code:", n_flag)
print(f"As % of final cohort: {n_flag / eligible_cohort['stay_id'].nunique() * 100:.1f}%")

esrd_final = aki_labels[aki_labels["stay_id"].isin(esrd_dialysis_stay_ids)]
if len(esrd_final) > 0:
    print(f"ESRD/dialysis AKI prevalence: {esrd_final['AKI'].mean()*100:.2f}% (n={len(esrd_final)})")
    print(f"Others AKI prevalence: {aki_labels[~aki_labels['stay_id'].isin(esrd_dialysis_stay_ids)]['AKI'].mean()*100:.2f}%")

In [ ]:
print(aki_labels.loc[aki_labels["on_crrt"]==True, "label_category"].value_counts())

In [ ]:
short_stay_final = eligible_cohort["stay_id"].isin(short_stay_ids)
print("Urine-evaluable rate among LOS<48h stays:", 
      aki_labels[aki_labels["stay_id"].isin(short_stay_ids)]["urine_evaluable"].mean())
print("Urine-evaluable rate among LOS>=48h stays:", 
      aki_labels[~aki_labels["stay_id"].isin(short_stay_ids)]["urine_evaluable"].mean())

In [ ]:
print("AKI prevalence, LOS<48h stays:", aki_labels[aki_labels["stay_id"].isin(short_stay_ids)]["AKI"].mean()*100)
print("AKI prevalence, LOS>=48h stays:", aki_labels[~aki_labels["stay_id"].isin(short_stay_ids)]["AKI"].mean()*100)

In [ ]:
print("Final cohort:", eligible_cohort["stay_id"].nunique())
print("AKI positive:", aki_labels["AKI"].sum())
print("AKI prevalence:", round(aki_labels["AKI"].mean()*100, 2))

In [ ]:
print(full_stats_df)
print("Combined kappa:", kappa_combined)

In [ ]:
current_fn_ids = set(compare_uo[(compare_uo["AKI_ref_uo"]==True) & (compare_uo["urine_labels"]==False)]["stay_id"])
print(f"Current FN count: {len(current_fn_ids)}")

# Check: are  of the current FN stays actually irrigant related
irrigant_stay_ids = set(irrigant_events["stay_id"].unique())
fn_irrigant_overlap = current_fn_ids & irrigant_stay_ids
print(f"Of {len(current_fn_ids)} FN stays, {len(fn_irrigant_overlap)} involve irrigant items at all")

# FN stays with weight=NaN
fn_weight_check = weight[weight["stay_id"].isin(current_fn_ids)]
fn_with_weight = set(fn_weight_check["stay_id"])
fn_without_weight = current_fn_ids - fn_with_weight
print(f"FN stays with NO weight at all: {len(fn_without_weight)} of {len(current_fn_ids)} ({len(fn_without_weight)/len(current_fn_ids)*100:.1f}%)")

In [ ]:
# final-cohort stays are NOT in the reference comparison
missing_from_compare = eligible_cohort["stay_id"].nunique() - len(compare)
print(f"Final cohort stays missing from combined comparison: {missing_from_compare}")
print(f"Final cohort stays missing from creatinine comparison: {eligible_cohort['stay_id'].nunique() - len(compare_creat)}")
print(f"Final cohort stays missing from urine comparison: {eligible_cohort['stay_id'].nunique() - len(compare_uo)}")

In [ ]:
# remaining negative urine_ml values in the raw data
print("Negative urine_ml rows remaining:", (outputevents["urine_ml"] < 0).sum())

# remaining negative computed RATES in the urine-evaluability function's output check
negative_rate_stays = []
for stay_id, group in outputevents[outputevents["stay_id"].isin(eligible_cohort["stay_id"])].groupby("stay_id"):
    wt_row = weight.loc[weight["stay_id"] == stay_id, "weight_kg"]
    if len(wt_row) == 0 or pd.isna(wt_row.values[0]):
        continue
    wt = wt_row.values[0]
    events = group[(group["hours_from_icu"] >= 24) & (group["hours_from_icu"] < 48)].sort_values("charttime")
    for _, row in events.iterrows():
        window = group[(group["charttime"] <= row["charttime"]) & (group["charttime"] >= row["charttime"] - pd.Timedelta(hours=6))]
        elapsed_hours = window["tm_since_last"].sum() / 60
        if elapsed_hours >= 6:
            rate = window["urine_ml"].sum() / wt / elapsed_hours
            if rate < 0:
                negative_rate_stays.append((stay_id, rate))

print(f"Stays with negative computed rate (post-fix): {len(negative_rate_stays)}")
print(negative_rate_stays[:10])

In [ ]:
# legitimate pairing check
neg_rows = outputevents[outputevents["urine_ml"] < 0]
print("Negative rows, itemid breakdown:")
print(neg_rows["itemid"].value_counts())
print("Negative rows, paired status:")
print(neg_rows["irrigant_paired"].value_counts())

In [ ]:
print("Unique stays with negative rate:", len(set(s for s, r in negative_rate_stays)))

#### 0–24h early-AKI exclusion and 24–48h outcome window Recheck

In [ ]:
# Leakage guardrail
early_full_timeline = generate_urine_labels_elapsed_time(
    outputevents[outputevents["stay_id"].isin(cohort["stay_id"])], weight, window_start=0, window_end=24
)
early_restricted_timeline = generate_urine_labels_elapsed_time(
    outputevents[(outputevents["stay_id"].isin(cohort["stay_id"])) & (outputevents["hours_from_icu"] < 24)],
    weight, window_start=0, window_end=24
)
merged_check = early_full_timeline.merge(early_restricted_timeline, on="stay_id", suffixes=("_full","_restricted"))
mismatches = merged_check[
    (merged_check["urine_evaluable_et_full"] != merged_check["urine_evaluable_et_restricted"]) |
    (merged_check["urine_labels_et_full"] != merged_check["urine_labels_et_restricted"])
]
print(f"Leakage guardrail: {len(mismatches)} mismatches found ")

In [ ]:
print("Overlap with ESRD set:", len(dialysis_stay_ids & esrd_stay_ids))
print("Dialysis-code stays NOT already ESRD:", len(dialysis_stay_ids - esrd_stay_ids))

final_combined = eligible_cohort["stay_id"].isin(esrd_dialysis_stay_ids).sum()
print(f"\nFinal cohort stays with ESRD OR dialysis-dependence: {final_combined}")

combined_final_aki = aki_labels[aki_labels["stay_id"].isin(esrd_dialysis_stay_ids)]
print(f"AKI prevalence, ESRD/dialysis group: {combined_final_aki['AKI'].mean()*100:.2f}% (n={len(combined_final_aki)})")
print(f"AKI prevalence, everyone else: {aki_labels[~aki_labels['stay_id'].isin(esrd_dialysis_stay_ids)]['AKI'].mean()*100:.2f}%")

In [ ]:
print("New cohort:", eligible_cohort["stay_id"].nunique())
print("New AKI positive:", aki_labels["AKI"].sum())
print("New AKI negative:", (~aki_labels["AKI"]).sum())
print("New AKI prevalence:", round(aki_labels["AKI"].mean()*100, 2))

print("\nBaseline source breakdown:")
print(baseline_summary)

print("\nDetection sources:")
print(source_summary)

print("\nFull diagnostics:")
print(full_stats_df)
print("Combined kappa:", kappa_combined)


In [ ]:
final_combined = eligible_cohort["stay_id"].isin(esrd_dialysis_stay_ids).sum()
print(f"Final cohort (v5) stays with ESRD/dialysis: {final_combined}")
combined_aki = aki_labels[aki_labels["stay_id"].isin(esrd_dialysis_stay_ids)]
print(f"AKI prevalence, ESRD/dialysis group: {combined_aki['AKI'].mean()*100:.2f}% (n={len(combined_aki)})")
print(f"AKI prevalence, everyone else: {aki_labels[~aki_labels['stay_id'].isin(esrd_dialysis_stay_ids)]['AKI'].mean()*100:.2f}%")

In [ ]:
print("Final cohort (v5) stays with LOS<48h:", eligible_cohort["stay_id"].isin(short_stay_ids).sum())
print("AKI prevalence, LOS<48h:", aki_labels[aki_labels["stay_id"].isin(short_stay_ids)]["AKI"].mean()*100)
print("AKI prevalence, LOS>=48h:", aki_labels[~aki_labels["stay_id"].isin(short_stay_ids)]["AKI"].mean()*100)

In [ ]:
missing_v5 = eligible_cohort["stay_id"].nunique() - len(compare)
print(f"Final cohort (v5) stays missing from comparison: {missing_v5}")
missing_ids_v5 = set(eligible_cohort["stay_id"]) - set(compare["stay_id"])
print(f"Also short-LOS: {len(missing_ids_v5 & short_stay_ids)} ({len(missing_ids_v5 & short_stay_ids)/len(missing_ids_v5)*100:.1f}%)")

In [ ]:
print("Stays on CRRT during 24-48h window:", aki_labels["on_crrt"].sum())
print("Of those, previously AKI-negative before CRRT rule applied:",
      (aki_labels["on_crrt"] & ~(aki_labels["aki_scr"] | aki_labels["urine_labels"])).sum())

compare["has_crrt"] = compare["stay_id"].isin(crrt_active_24_48)
print("CRRT stays in comparison population:", compare["has_crrt"].sum())
print("Agreement excluding CRRT stays:", round((compare.loc[~compare["has_crrt"],"AKI"]==compare.loc[~compare["has_crrt"],"AKI_ref"]).mean(),4))
print("Agreement among CRRT stays only:", round((compare.loc[compare["has_crrt"],"AKI"]==compare.loc[compare["has_crrt"],"AKI_ref"]).mean(),4))

In [ ]:
print(f"Stays covered by mimiciv_derived.weight_durations: {len(ref_weight_stays)}")
print(f"Stays covered by our first-24h weight table: {len(own_weight_stays)}")
print(f"Overlap: {len(ref_weight_stays & own_weight_stays)}")

# Re-check final-cohort weight availability specifically
weight_available_v5 = weight[weight["stay_id"].isin(eligible_cohort["stay_id"])]["stay_id"].nunique()
print(f"Final cohort (current) weight available: {weight_available_v5} of {eligible_cohort['stay_id'].nunique()} "
      f"({weight_available_v5/eligible_cohort['stay_id'].nunique()*100:.1f}%)")

In [ ]:
if (OUTPUT / "labels_v5.parquet").exists():
       _prev = pd.read_parquet(OUTPUT / "labels_v5.parquet")     
       _now = aki_labels.assign(short_los=aki_labels["stay_id"].isin(short_stay_ids),
                                   has_esrd=aki_labels["stay_id"].isin(esrd_dialysis_stay_ids))
       cols = ["aki_0_3","aki_1_5x","urine_labels","aki_scr","urine_evaluable","on_crrt",
              "short_los","has_esrd","post_outtime_data","AKI"]
       _m = _prev[["stay_id"]+cols].merge(_now[["stay_id"]+cols], on="stay_id", suffixes=("_old","_new"))
       assert len(_m) == len(_prev) == len(_now), "stay set changed"
       for c in cols:
              assert (_m[c+"_old"].astype(bool) == _m[c+"_new"].astype(bool)).all(), f"{c} changed vs frozen v5"
       print("Rebuilt labels match the frozen labels_v5 on all columns")
else:
      print("No frozen labels_v5 yet; they will be created below.")

In [ ]:

aki_labels["short_los"] = aki_labels["stay_id"].isin(short_stay_ids)
aki_labels["has_esrd"] = aki_labels["stay_id"].isin(esrd_dialysis_stay_ids)

FINAL_LABELS_PATH = OUTPUT / "labels_v5.parquet"
labels_final = aki_labels[["stay_id","aki_0_3","aki_1_5x","urine_labels","aki_scr",
                            "urine_evaluable","on_crrt","label_category",
                            "short_los","has_esrd","post_outtime_data","AKI"]].copy()
if FINAL_LABELS_PATH.exists():
       with open(OUTPUT / "labels_v5.metadata.json") as f:
           meta = json.load(f)
       current = hashlib.sha256(FINAL_LABELS_PATH.read_bytes()).hexdigest()
       assert current == meta["checksum_sha256"], "frozen labels_v5.parquet has been modified"
       print("labels_v5 already frozen, checksum verified, not rewritten:", current)
else:
       labels_final.to_parquet(FINAL_LABELS_PATH, index=False)
       file_hash = hashlib.sha256(FINAL_LABELS_PATH.read_bytes()).hexdigest()
       metadata = {"version": "v5", "row_count": int(len(labels_final)), "checksum_sha256": file_hash}
       with open(OUTPUT / "labels_v5.metadata.json", "w") as f:
           json.dump(metadata, f, indent=2)
       print("Labels frozen:", metadata)

In [ ]:
run_log = {"date": str(datetime.date.today()), "cohort_n": int(len(cohort)),
              "final_n": int(len(labels_final)), "aki_positive": int(labels_final["AKI"].sum()),
              "early_excluded": int(len(early_aki_ids)), "urine_evaluable": int(labels_final["urine_evaluable"].sum()),
              "labels_v5_sha256": hashlib.sha256(FINAL_LABELS_PATH.read_bytes()).hexdigest()}
with open(OUTPUT / "run_log_primary.json", "w") as f:
       json.dump(run_log, f, indent=2)

In [ ]:
EXPECTED = {"final_n": 28977, "aki_positive": 7624, "urine_evaluable": 17045, "early_excluded": 22829}
got = {k: run_log[k] for k in EXPECTED} 
assert got == EXPECTED, f"Reproduction mismatch: {got} vs {EXPECTED}"
print("Reproduced the published V5 counts")